# Chapter 18 - Reinforcement Learning

## 1. Reinforcement Learning Basics, Policy Search, and OpenAI Gym

Reinforcement Learning (RL) memformalkan proses pengambilan keputusan sebagai interaksi berulang antara sebuah **agen** dan **lingkungan (environment)**. Pada setiap langkah waktu, agen mengamati keadaan (state), memilih sebuah aksi (action), lalu menerima umpan balik berupa **reward** dan transisi ke state berikutnya. Tujuan utama agen adalah memaksimalkan **expected cumulative discounted reward** dalam jangka panjang.

Kerangka ini dapat diterapkan pada berbagai masalah nyata, seperti pengendalian robot, permainan Atari (misalnya *Ms. Pac-Man*), permainan strategi kompleks seperti Go, pengaturan suhu otomatis, hingga sistem trading algoritmik. Perbedaan antar masalah terletak pada definisi state, ruang aksi, dinamika environment, dan fungsi reward, tetapi struktur interaksi agen–lingkungan tetap sama.

Perilaku agen ditentukan oleh sebuah **policy**, yaitu aturan yang memetakan observasi atau state ke aksi. Policy dapat bersifat deterministik maupun stokastik; dalam praktik modern, policy sering dimodelkan sebagai neural network yang menghasilkan distribusi probabilitas atas aksi yang mungkin. Pendekatan untuk menemukan policy optimal dikenal sebagai **policy search**, yang mencakup metode sederhana seperti pencarian brute-force, pendekatan evolusioner (misalnya genetic algorithms dan NEAT), hingga metode optimasi berbasis gradien yang langsung menyesuaikan parameter policy.

Untuk eksperimen dan pembelajaran, **OpenAI Gym** menyediakan kumpulan environment standar dengan antarmuka seragam. Salah satu contoh populer adalah *CartPole-v1*, yang mensimulasikan kereta dengan tiang yang harus dijaga tetap seimbang. Environment ini menyediakan metode `reset()` dan `step(action)`, serta atribut `observation_space` dan `action_space` untuk mendeskripsikan dimensi state dan aksi.


## **Example: Menggunakan OpenAI Gym dan Policy Sederhana di CartPole**

In [ ]:
import gym
import numpy as np

env = gym.make("CartPole-v1")
obs = env.reset()

def basic_policy(obs):
    angle = obs[2]
    return 0 if angle < 0 else 1  # 0: left, 1: right

totals = []
for episode in range(500):
    episode_rewards = 0
    obs = env.reset()
    for step in range(200):
        action = basic_policy(obs)
        obs, reward, done, info = env.step(action)
        episode_rewards += reward
        if done:
            break
    totals.append(episode_rewards)

print(np.mean(totals), np.std(totals), np.min(totals), np.max(totals))
env.close()


## 2. Policy Gradients and CartPole with TensorFlow/Keras

Metode **Policy Gradient (PG)** mengoptimasi policy secara langsung dengan memaksimalkan ekspektasi return, tanpa membangun model eksplisit dari environment atau fungsi nilai. Ide dasarnya adalah memperkuat aksi yang menghasilkan return tinggi dan melemahkan aksi yang menghasilkan return rendah. Proses training umumnya melibatkan pengumpulan beberapa episode, perhitungan return terdiskonto untuk setiap aksi, dan penggunaan nilai tersebut sebagai sinyal pembelajaran.

Algoritma klasik **REINFORCE** mengimplementasikan pendekatan ini dengan mendefinisikan loss sebagai negatif dari log-probabilitas aksi yang diambil, dikalikan dengan return yang diperoleh. Gradien dari loss ini menjadi estimasi gradien reward terhadap parameter policy, sehingga update parameter mendorong peningkatan probabilitas aksi yang mengarah pada return tinggi.

Pada environment CartPole, policy dapat dimodelkan sebagai neural network sederhana yang menerima vektor observasi berdimensi empat dan menghasilkan probabilitas untuk salah satu aksi (misalnya mendorong ke kiri), biasanya melalui fungsi aktivasi sigmoid. Aksi kemudian di-*sample* secara stokastik dari distribusi ini, yang secara alami mendorong eksplorasi.

Return dihitung sebagai jumlah reward terdiskonto dengan faktor diskonto \( \gamma \) (umumnya sekitar 0.95), lalu dinormalisasi di seluruh aksi dan episode untuk mengurangi varians estimasi gradien. Normalisasi ini menghasilkan **advantage** yang lebih stabil, sehingga proses pembelajaran menjadi lebih konsisten dan konvergen.


## **Example: Menggunakan OpenAI Gym dan Policy Sederhana di CartPole**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import gym

env = gym.make("CartPole-v1")
n_inputs = env.observation_space.shape[0]

model = keras.models.Sequential([
    keras.layers.Dense(5, activation="elu", input_shape=[n_inputs]),
    keras.layers.Dense(1, activation="sigmoid"),
])

optimizer = keras.optimizers.Adam(learning_rate=0.01)
loss_fn = keras.losses.binary_crossentropy

def play_one_step(env, obs, model, loss_fn):
    with tf.GradientTape() as tape:
        left_proba = model(obs[np.newaxis])
        action = tf.cast(tf.random.uniform([1, 1]) > left_proba, tf.int32)
        y_target = 1.0 - tf.cast(action, tf.float32)
        loss = tf.reduce_mean(loss_fn(y_target, left_proba))
    grads = tape.gradient(loss, model.trainable_variables)
    obs, reward, done, info = env.step(int(action[0, 0]))
    return obs, reward, done, grads

def play_multiple_episodes(env, n_episodes, n_max_steps, model, loss_fn):
    all_rewards, all_grads = [], []
    for episode in range(n_episodes):
        current_rewards, current_grads = [], []
        obs = env.reset()
        for step in range(n_max_steps):
            obs, reward, done, grads = play_one_step(env, obs, model, loss_fn)
            current_rewards.append(reward)
            current_grads.append(grads)
            if done:
                break
        all_rewards.append(current_rewards)
        all_grads.append(current_grads)
    return all_rewards, all_grads

def discount_rewards(rewards, discount_factor):
    discounted = np.array(rewards, dtype=np.float32)
    for step in range(len(rewards) - 2, -1, -1):
        discounted[step] += discount_factor * discounted[step + 1]
    return discounted

def discount_and_normalize_rewards(all_rewards, discount_factor):
    all_discounted = [discount_rewards(r, discount_factor) for r in all_rewards]
    flat = np.concatenate(all_discounted)
    mean, std = flat.mean(), flat.std()
    return [(d - mean) / (std + 1e-8) for d in all_discounted]

n_iterations = 150
n_episodes_per_update = 10
n_max_steps = 200
discount_factor = 0.95

for iteration in range(n_iterations):
    all_rewards, all_grads = play_multiple_episodes(
        env, n_episodes_per_update, n_max_steps, model, loss_fn
    )
    all_final_rewards = discount_and_normalize_rewards(
        all_rewards, discount_factor
    )
    all_mean_grads = []
    for var_index in range(len(model.trainable_variables)):
        mean_grads = tf.reduce_mean([
            final_reward * all_grads[episode_idx][step][var_index]
            for episode_idx, final_rewards in enumerate(all_final_rewards)
            for step, final_reward in enumerate(final_rewards)
        ], axis=0)
        all_mean_grads.append(mean_grads)
    optimizer.apply_gradients(zip(all_mean_grads, model.trainable_variables))

env.close()


## 3. Markov Decision Processes, Temporal-Difference Learning, Q-Learning, and Deep Q-Networks

Sebagian besar permasalahan Reinforcement Learning dapat diformalkan sebagai **Markov Decision Process (MDP)**, yang didefinisikan oleh himpunan state, himpunan aksi, probabilitas transisi $$T(s,a,s')$$, fungsi reward $$R(s,a,s')$$, serta faktor diskonto $$\gamma \in [0,1]$$. Asumsi Markov menyatakan bahwa transisi ke state berikutnya dan reward hanya bergantung pada state dan aksi saat ini, bukan pada riwayat sebelumnya.

Dalam kerangka MDP, **Bellman Optimality Equation** mendefinisikan nilai optimal dari suatu state $$V^*(s)$$ dan nilai optimal pasangan state–aksi $$Q^*(s,a)$$. Jika fungsi transisi dan reward diketahui secara eksplisit, nilai optimal ini dapat dihitung secara iteratif menggunakan algoritma **Value Iteration** atau **Q-Value Iteration**, yang berulang kali menerapkan persamaan Bellman hingga konvergen.

Namun, dalam sebagian besar skenario nyata, agen tidak memiliki akses langsung ke $$T$$ dan $$R$$. Oleh karena itu, pembelajaran dilakukan secara *model-free* melalui pengalaman interaksi langsung dengan environment. **Temporal-Difference (TD) Learning** menggabungkan ide Monte Carlo dan dynamic programming dengan memperbarui estimasi nilai berdasarkan perbedaan antara prediksi saat ini dan target *bootstrap* dari estimasi berikutnya.

Salah satu algoritma TD yang paling dikenal adalah **Q-Learning**, yang memperbarui estimasi nilai aksi–state menggunakan aturan pembaruan:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \bigl(r + \gamma \max_{a'} Q(s',a') - Q(s,a)\bigr)
$$

di mana $$\alpha$$ adalah *learning rate*. Q-Learning bersifat *off-policy*, karena proses pembelajarannya menuju policy optimal terlepas dari policy eksplorasi yang sedang dijalankan.

Untuk masalah dengan ruang state yang sangat besar atau kontinu, representasi tabel Q tidak lagi praktis. Dalam kasus ini digunakan **Approximate Q-Learning**, di mana fungsi nilai aksi didekati oleh fungsi parametrik $$Q_\theta(s,a)$$. Ketika fungsi aproksimasi tersebut direpresentasikan oleh neural network dalam, pendekatan ini dikenal sebagai **Deep Q-Network (DQN)**.

DQN dilatih dengan meminimalkan *Mean Squared Error* antara prediksi nilai Q dan target TD berikut:

$$
y = r + \gamma \max_{a'} Q_\theta(s',a')
$$

Pendekatan ini memungkinkan agen menangani environment berdimensi tinggi, seperti input visual pada permainan Atari, dan menjadi tonggak penting dalam perkembangan Reinforcement Learning modern.


## **Example: Tabular Q-Learning dan DQN untuk CartPole**

In [ ]:
import numpy as np

# Tabular example on tiny MDP (3 states, handcrafted transitions)
transition_probabilities = [
    [[0.7, 0.3, 0.0], [1.0, 0.0, 0.0], [0.8, 0.2, 0.0]],
    [[0.0, 1.0, 0.0], None,           [0.0, 0.0, 1.0]],
    [None,           [0.8, 0.1, 0.1], None          ],
]
rewards = [
    [[+10, 0, 0], [0, 0, 0], [0, 0, 0]],
    [[0, 0, 0],   [0, 0, 0], [0, 0, -50]],
    [[0, 0, 0],   [+40, 0, 0], [0, 0, 0]],
]
possible_actions = [[0, 1, 2], [0, 2], [1]]

Q_values = np.full((3, 3), -np.inf)
for s, actions in enumerate(possible_actions):
    Q_values[s, actions] = 0.0

def step(state, action):
    probas = transition_probabilities[state][action]
    next_state = np.random.choice([0, 1, 2], p=probas)
    reward = rewards[state][action][next_state]
    return next_state, reward

def exploration_policy(state):
    return np.random.choice(possible_actions[state])

alpha0, decay, gamma = 0.05, 0.005, 0.90
state = 0
for iteration in range(10000):
    action = exploration_policy(state)
    next_state, reward = step(state, action)
    next_value = np.max(Q_values[next_state])
    alpha = alpha0 / (1 + iteration * decay)
    Q_values[state, action] *= (1 - alpha)
    Q_values[state, action] += alpha * (reward + gamma * next_value)
    state = next_state

optimal_actions = np.argmax(Q_values, axis=1)
print("Optimal actions per state:", optimal_actions)


In [ ]:
# DQN for CartPole
import gym
import numpy as np
import tensorflow as tf
from tensorflow import keras
from collections import deque

env = gym.make("CartPole-v0")
input_shape = [env.observation_space.shape[0]]
n_outputs = env.action_space.n

model = keras.models.Sequential([
    keras.layers.Dense(32, activation="elu", input_shape=input_shape),
    keras.layers.Dense(32, activation="elu"),
    keras.layers.Dense(n_outputs),
])

def epsilon_greedy_policy(state, epsilon=0):
    if np.random.rand() < epsilon:
        return np.random.randint(n_outputs)
    Q_values = model.predict(state[np.newaxis], verbose=0)
    return int(np.argmax(Q_values[0]))

replay_buffer = deque(maxlen=2000)

def sample_experiences(batch_size):
    indices = np.random.randint(len(replay_buffer), size=batch_size)
    batch = [replay_buffer[i] for i in indices]
    states, actions, rewards, next_states, dones = [
        np.array([exp[field] for exp in batch]) for field in range(5)
    ]
    return states, actions, rewards, next_states, dones

def play_one_step(env, state, epsilon):
    action = epsilon_greedy_policy(state, epsilon)
    next_state, reward, done, info = env.step(action)
    replay_buffer.append((state, action, reward, next_state, done))
    return next_state, reward, done, info

batch_size = 32
discount_factor = 0.95
optimizer = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn = keras.losses.mean_squared_error

def training_step(batch_size):
    states, actions, rewards, next_states, dones = sample_experiences(batch_size)
    next_Q_values = model.predict(next_states, verbose=0)
    max_next_Q_values = np.max(next_Q_values, axis=1)
    target_Q_values = rewards + (1 - dones) * discount_factor * max_next_Q_values
    mask = tf.one_hot(actions, n_outputs)
    with tf.GradientTape() as tape:
        all_Q = model(states)
        Q_values = tf.reduce_sum(all_Q * mask, axis=1, keepdims=True)
        loss = tf.reduce_mean(loss_fn(target_Q_values, Q_values))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

for episode in range(600):
    obs = env.reset()
    for step in range(200):
        epsilon = max(1 - episode / 500, 0.01)
        obs, reward, done, info = play_one_step(env, obs, epsilon)
        if done:
            break
    if episode > 50:
        training_step(batch_size)

env.close()


## 4. Deep Q-Learning Variants and Improvements

Meskipun **Deep Q-Network (DQN)** mampu menangani ruang state berdimensi tinggi, pendekatan dasar ini sering tidak stabil dan rentan terhadap *catastrophic forgetting*. Penyebab utamanya adalah karena model secara simultan memperbarui parameter jaringan sekaligus menggunakan prediksinya sendiri sebagai target pembelajaran, sementara distribusi data berubah seiring dengan perubahan policy.

Untuk mengatasi masalah tersebut, berbagai varian DQN dikembangkan guna meningkatkan stabilitas dan kinerja pembelajaran, antara lain sebagai berikut.

**Fixed Q-Value Targets.**  
Pendekatan ini menggunakan dua jaringan terpisah, yaitu *online network* dan *target network*. Online network digunakan untuk pembelajaran, sedangkan target network hanya dipakai untuk menghasilkan target Q-value. Bobot target network diperbarui secara periodik (misalnya setiap 10.000 langkah), sehingga target pembelajaran menjadi lebih stabil dan mengurangi osilasi selama training.

**Double DQN.**  
Double DQN diperkenalkan untuk mengurangi *over-estimation bias* yang muncul akibat penggunaan operator maksimum pada estimasi Q-value. Dalam pendekatan ini, online network digunakan untuk memilih aksi terbaik pada state berikutnya, sementara target network digunakan untuk mengevaluasi nilai Q dari aksi tersebut. Pemisahan proses pemilihan dan evaluasi ini terbukti meningkatkan akurasi estimasi nilai.

**Prioritized Experience Replay (PER).**  
Alih-alih mengambil sampel pengalaman secara uniform dari replay buffer, PER memberikan probabilitas sampling lebih tinggi pada transisi dengan *temporal-difference (TD) error* besar, karena pengalaman tersebut dianggap lebih informatif atau “mengejutkan”. Untuk menjaga estimator tetap tidak bias, digunakan *importance-sampling weights* yang menyesuaikan kontribusi tiap sampel terhadap loss.

**Dueling DQN.**  
Dueling DQN memodifikasi arsitektur jaringan dengan memisahkan estimasi **state value** dan **advantage function** untuk tiap aksi. Kedua komponen ini kemudian digabungkan untuk menghasilkan Q-value. Pendekatan ini sangat membantu pada state di mana pilihan aksi memiliki dampak yang relatif kecil, sehingga jaringan dapat lebih efisien mempelajari nilai state secara umum.


## **Example: Double DQN dan Dueling Architecture**

In [ ]:
# Double DQN training_step (modifikasi target)
target = keras.models.clone_model(model)
target.set_weights(model.get_weights())

def training_step_double(batch_size):
    states, actions, rewards, next_states, dones = sample_experiences(batch_size)
    next_Q_online = model.predict(next_states, verbose=0)
    best_next_actions = np.argmax(next_Q_online, axis=1)
    next_Q_target = target.predict(next_states, verbose=0)
    next_mask = tf.one_hot(best_next_actions, n_outputs).numpy()
    next_best_Q = np.sum(next_Q_target * next_mask, axis=1)
    target_Q_values = rewards + (1 - dones) * discount_factor * next_best_Q
    mask = tf.one_hot(actions, n_outputs)
    with tf.GradientTape() as tape:
        all_Q = model(states)
        Q_values = tf.reduce_sum(all_Q * mask, axis=1, keepdims=True)
        loss = tf.reduce_mean(loss_fn(target_Q_values, Q_values))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

# Periodically sync target network
if episode % 50 == 0:
    target.set_weights(model.get_weights())

# Dueling DQN architecture for CartPole-like inputs
from tensorflow import keras
import tensorflow.keras.backend as K

n_outputs = 2
input_states = keras.layers.Input(shape=[4])
hidden1 = keras.layers.Dense(32, activation="elu")(input_states)
hidden2 = keras.layers.Dense(32, activation="elu")(hidden1)
state_values = keras.layers.Dense(1)(hidden2)
raw_advantages = keras.layers.Dense(n_outputs)(hidden2)
advantages = raw_advantages - K.max(raw_advantages, axis=1, keepdims=True)
Q_values = state_values + advantages
dueling_model = keras.Model(inputs=[input_states], outputs=[Q_values])


## 5. TF-Agents, Atari Breakout, and Modern RL Algorithms

Bagian akhir bab memperkenalkan **TF-Agents**, sebuah library Reinforcement Learning berbasis TensorFlow yang menyediakan *environment wrapper*, *replay buffer* yang efisien, serta implementasi berbagai algoritma RL modern seperti **DQN, Double DQN, REINFORCE, PPO, dan Soft Actor-Critic (SAC)**. Selain itu, TF-Agents menyediakan *driver utilities* untuk mengotomatiskan proses pengumpulan *experience* dari environment.

Sebagai contoh utama, bab ini membahas pelatihan agen **DQN** pada game **Atari Breakout-v4** dari OpenAI Gym. Proses ini mengikuti praktik standar dari paper DeepMind (2015), termasuk *preprocessing* frame (konversi ke grayscale, downsampling resolusi, dan *frame stacking*), penggunaan replay buffer berukuran besar, target network terpisah, serta pembaruan parameter secara berkala untuk menjaga stabilitas training.

Environment dalam TF-Agents mengembalikan objek **TimeStep** yang berisi `step_type`, `reward`, `discount`, dan `observation`. Library ini juga mendefinisikan spesifikasi formal melalui `observation_spec`, `action_spec`, dan `time_step_spec`, yang memudahkan validasi bentuk data dan integrasi dengan berbagai algoritma.

Pengumpulan data dilakukan menggunakan **DynamicStepDriver** atau **DynamicEpisodeDriver**, yang menjalankan policy pada environment, meneruskan aksi, menyimpan *trajectory* ke replay buffer, serta memperbarui metrik pelatihan secara otomatis. Pendekatan ini memisahkan logika interaksi environment dari logika pembelajaran, sehingga kode menjadi lebih modular dan terstruktur.

Selain DQN, bab ini juga mengulas secara singkat beberapa algoritma RL modern:

- **Actor–Critic (A2C/A3C)**, yang memisahkan policy (actor) dan estimasi nilai (critic) untuk mengurangi varians gradien.
- **Soft Actor-Critic (SAC)**, yang mengoptimalkan kombinasi reward dan entropi aksi, sehingga menghasilkan policy yang lebih stabil dan eksploratif.
- **Proximal Policy Optimization (PPO)**, yang menggunakan *clipped loss* untuk membatasi perubahan policy dan meningkatkan stabilitas training.
- **Curiosity-driven exploration**, yang menambahkan reward intrinsik berbasis ketidakpastian atau error prediksi untuk mendorong eksplorasi pada environment dengan reward jarang.

Pendekatan-pendekatan ini merepresentasikan arah utama Reinforcement Learning modern, dengan fokus pada stabilitas, efisiensi sampel, dan skalabilitas ke environment kompleks.


## **Example: Setup TF‑Agents DQN untuk Breakout**

In [ ]:
import tensorflow as tf
from tf_agents.environments import suite_gym
from tf_agents.environments.wrappers import ActionRepeat
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.agents.dqn import dqn_agent
from tf_agents.networks import q_network
from tf_agents.drivers.dynamic_step_driver import DynamicStepDriver
from tf_agents.policies.random_tf_policy import RandomTFPolicy
from tf_agents.utils.common import function

# Load Breakout environment via TF-Agents (wrapper around Gym)
tf_env = suite_gym.load("Breakout-v4")

# Q-network (CNN) on preprocessed 84x84x4 observations (assume preprocessing wrapper applied)
preproc_env = tf_env  # placeholder if you add preprocessing wrappers
q_net = q_network.QNetwork(
    preproc_env.observation_spec(),
    preproc_env.action_spec(),
    fc_layer_params=(512,)
)

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
train_step = tf.Variable(0)

agent = dqn_agent.DqnAgent(
    preproc_env.time_step_spec(),
    preproc_env.action_spec(),
    q_network=q_net,
    optimizer=optimizer,
    td_errors_loss_fn=tf.keras.losses.Huber(reduction="none"),
    train_step_counter=train_step,
)
agent.initialize()

replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    data_spec=agent.collect_data_spec,
    batch_size=preproc_env.batch_size,
    max_length=1000000
)
replay_buffer_observer = replay_buffer.add_batch

update_period = 4
collect_driver = DynamicStepDriver(
    preproc_env,
    agent.collect_policy,
    observers=[replay_buffer_observer],
    num_steps=update_period
)

# Warm up replay buffer with random policy
initial_collect_policy = RandomTFPolicy(preproc_env.time_step_spec(),
                                        preproc_env.action_spec())
init_driver = DynamicStepDriver(
    preproc_env,
    initial_collect_policy,
    observers=[replay_buffer.add_batch],
    num_steps=20000
)
init_driver.run()

dataset = replay_buffer.as_dataset(
    sample_batch_size=64,
    num_steps=2,
    num_parallel_calls=3
).prefetch(3)
iterator = iter(dataset)

collect_driver.run = function(collect_driver.run)
agent.train = function(agent.train)

def train_agent(n_iterations):
    time_step = None
    policy_state = agent.collect_policy.get_initial_state(preproc_env.batch_size)
    for iteration in range(n_iterations):
        time_step, policy_state = collect_driver.run(time_step, policy_state)
        trajectories, buffer_info = next(iterator)
        train_loss = agent.train(trajectories)

# Example (would take long in practice):
# train_agent(1000000)
